In [ ]:
# import util functions from external util file
%run util.py

In [ ]:
# Set to True if you want to open excel files through xlwings in the background
# Note: This starts excel. Excel is required on your machine
if False:
    app = xw.App(visible=True)
else:
    app = xw.App()

In [ ]:
#plt.style.use('seaborn-v0_8-white')

In [ ]:
plt.rcParams.update({'font.size': 14})  

NOTE: This notebook expects that the monte carlo simulation is already completed and the corresponding rows exist in the dataframe. You need to save the file after you run the MC Simulation!

Make sure you change the file name below to address the file you created through the MC simulation!

In [ ]:
# Setup

# Path to the data file 
# THIS LINE NEEDS TO BE ADAPTED
excel_file = './methodology_data_MC_20250409_105502.xlsx' 

 # Name of the sheet containing data
sheet_name = 'data' 

num_header_rows = 1 # number of rows with metadata or headers
num_header_columns = 3 # number of columns with metadata or headers

# Connect to Excel
# THIS OPENS AND EDITS THE EXCEL FILE ON YOUR COMPUTER!
wb = app.books.open(excel_file)
sheet = wb.sheets[sheet_name]



# Determine the range of columns and results
data_range = sheet.range("A1").expand()  # Automatically detect data range
num_columns = data_range.columns.count  # Total number of columns
num_rows = data_range.rows.count  # Total number of columns

### Overcrediting by Year

In [ ]:
num_columns

In [ ]:
# Loading data from excel into arrays
# (Yes, this could be improved a lot)

row_monitoring_period_raw_file_non_empty_cells = get_row_by_variable("monitoring_period_raw_file_non_empty_cells", sheet, num_rows) 
row_year = get_row_by_variable("year", sheet, num_rows) 
row_numeric_id = get_row_by_variable("numeric_id", sheet, num_rows) 
row_platform = get_row_by_variable("platform", sheet, num_rows) 
row_ER_mc_functionality_factor = get_row_by_variable("ER_mc_functionality_factor", sheet, num_rows) 
row_ER_mc_fnrb_factor = get_row_by_variable("ER_mc_fnrb_factor", sheet, num_rows) 
row_AMS_III_R = get_row_by_variable("AMS-III.R", sheet, num_rows) 
row_AMS_I_E = get_row_by_variable("AMS-I.E", sheet, num_rows) 
row_GS_TPDDTEC = get_row_by_variable("GS TPDDTEC", sheet, num_rows) 
row_ER_per_total_HH = get_row_by_variable("ER_per_total_HH", sheet, num_rows) 
row_fNRB = get_row_by_variable("f_NRB", sheet, num_rows) 



file_non_empty_cells = sheet.range((row_monitoring_period_raw_file_non_empty_cells, num_header_columns + 1), (row_monitoring_period_raw_file_non_empty_cells, num_columns)).value
year = sheet.range((row_year, num_header_columns + 1), (row_year, num_columns)).value
numeric_id = sheet.range((row_numeric_id, num_header_columns + 1), (row_numeric_id, num_columns)).value
platform = sheet.range((row_platform, num_header_columns + 1), (row_platform, num_columns)).value
ER_mc_functionality_factor = sheet.range((row_ER_mc_functionality_factor, num_header_columns + 1), (row_ER_mc_functionality_factor, num_columns)).value
ER_mc_fnrb_factor = sheet.range((row_ER_mc_fnrb_factor, num_header_columns + 1), (row_ER_mc_fnrb_factor, num_columns)).value
AMS_III_R = sheet.range((row_AMS_III_R, num_header_columns + 1), (row_AMS_III_R, num_columns)).value
AMS_I_E = sheet.range((row_AMS_I_E, num_header_columns + 1), (row_AMS_I_E, num_columns)).value
GS_TPDDTEC = sheet.range((row_GS_TPDDTEC, num_header_columns + 1), (row_GS_TPDDTEC, num_columns)).value
ER_per_total_HH = sheet.range((row_ER_per_total_HH, num_header_columns + 1), (row_ER_per_total_HH, num_columns)).value
fNRB = sheet.range((row_fNRB, num_header_columns + 1), (row_fNRB, num_columns)).value

In [ ]:
row_ER_per_total_HH

In [ ]:

df = pd.DataFrame({
    'Year': year,
    'Number of non-empty cells': file_non_empty_cells,
    'Numeric id': numeric_id,
    'Platform': platform,
    'ER_mc_functionality_factor': ER_mc_functionality_factor,
    'ER_mc_fnrb_factor': ER_mc_fnrb_factor,
    'AMS_III_R': AMS_III_R
})

df = df.dropna(subset=[
    'Year',
    'Number of non-empty cells',
    'ER_mc_functionality_factor',
    'ER_mc_fnrb_factor'
])

fig, ax1 = plt.subplots()

# === Left Y-axis: 'Number of non-empty cells' ===
x = df['Year']
y1 = df['Number of non-empty cells']
coeffs1 = np.polyfit(x, y1, deg=1)
reg_line1 = np.poly1d(coeffs1)

# Scatter + regression (left)
#ax1.scatter(x, y1, color='blue', label='Non-empty cells')
ax1.plot(np.linspace(x.min(), x.max(), 100),
         reg_line1(np.linspace(x.min(), x.max(), 100)),
         color='black', linestyle='--', label='Regression Non-empty Cells')

ax1.set_xlabel('Year')
ax1.set_ylabel('Number of Non-empty Cells', color='black')
ax1.tick_params(axis='y', labelcolor='black')


# === Right Y-axis: 'ER_mc_functionality_factor' ===
ax2 = ax1.twinx()

y2 = df['ER_mc_functionality_factor']
coeffs2 = np.polyfit(x, y2, deg=1)
reg_line2 = np.poly1d(coeffs2)

# Scatter + regression (right)
#ax2.scatter(x, y2, color='red', label='ER_mc_factor')
ax2.plot(np.linspace(x.min(), x.max(), 100),
         reg_line2(np.linspace(x.min(), x.max(), 100)),
         color='blue', linestyle='--', label='Regression Factor Functionality')

ax2.tick_params(axis='y', labelcolor='blue')

# === Right Y-axis: 'ER_mc_fnrb_factor' ===

# drop projects with AMS-R.III
df = df[
    df['AMS_III_R'] != 1
]


x2 = df['Year']
y3 = df['ER_mc_fnrb_factor']
coeffs3 = np.polyfit(x2, y3, deg=1)
reg_line3 = np.poly1d(coeffs3)

# Scatter + regression (right)
#ax2.scatter(x2, y2, color='red', label='ER_mc_factor')
ax2.plot(np.linspace(x2.min(), x2.max(), 100),
         reg_line3(np.linspace(x2.min(), x2.max(), 100)),
         color='red', linestyle='--', label='Regression Factor f_NRB')

ax1.set_ylim(bottom=0)  # Left y-axis starts at 0
ax2.set_ylim(bottom=0)  # Right y-axis starts at 0
ax2.set_ylim(top=6)  # Right y-axis starts at 0

# Collect line handles and labels from both axes
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()

# Combine and create a single legend
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper left')

ax2.set_ylabel('Over-crediting factor', color='black')
ax2.tick_params(axis='y', labelcolor='black')

# Title and layout
plt.title('Regression Lines: Non-empty Cells vs Over-crediting factors')
fig.tight_layout()
plt.show()

### Over-crediting factors over time by kind of factor and methodology

In [ ]:
df = pd.DataFrame({
    'Year': year, #int
    'Number of non-empty cells': file_non_empty_cells, #int
    'Numeric id': numeric_id, #int
    'Platform': platform, #string identifier
    'ER_mc_functionality_factor': ER_mc_functionality_factor,  #floating point
    'ER_mc_fnrb_factor': ER_mc_fnrb_factor, #floating point
    'AMS_III_R': AMS_III_R, #binary
    'AMS_I_E': AMS_I_E, #binary
    'GS_TPDDTEC': GS_TPDDTEC #binary
})

df = df.dropna(subset=[
    'Year',
    'Number of non-empty cells',
    'ER_mc_functionality_factor',
    'ER_mc_fnrb_factor'
])

# Filter by methodology
df_iii_r = df[df['AMS_III_R'] == 1]
df_i_e = df[df['AMS_I_E'] == 1]
df_gs = df[df['GS_TPDDTEC'] == 1]


def plot_regression_line(x, y, color, label):
    x_reshape = np.array(x).reshape(-1, 1)
    y_reshape = np.array(y).reshape(-1, 1)
    model = LinearRegression()
    model.fit(x_reshape, y_reshape)
    y_pred = model.predict(x_reshape)
    # Sort for cleaner line
    sorted_indices = np.argsort(x)
    plt.plot(np.array(x)[sorted_indices], y_pred[sorted_indices], color=color, linewidth=1, alpha=0.3, label=label)

# Define colors for consistent use
colors = {
    'AMS_III_R': 'tab:blue',
    'AMS_I_E': 'tab:orange',
    'GS_TPDDTEC': 'tab:green'
}


# Create scatter plot
plt.figure(figsize=(12, 6))

# Scatter + regression for functionality factor
plt.scatter(df_iii_r['Year'], df_iii_r['ER_mc_functionality_factor'], label='AMS_III_R', alpha=0.6, color=colors['AMS_III_R'])
plot_regression_line(df_iii_r['Year'], df_iii_r['ER_mc_functionality_factor'], color=colors['AMS_III_R'], label=None)

plt.scatter(df_i_e['Year'], df_i_e['ER_mc_functionality_factor'], label='AMS_I_E', alpha=0.6, color=colors['AMS_I_E'])
plot_regression_line(df_i_e['Year'], df_i_e['ER_mc_functionality_factor'], color=colors['AMS_I_E'], label=None)

plt.scatter(df_gs['Year'], df_gs['ER_mc_functionality_factor'], label='GS_TPDDTEC', alpha=0.6, color=colors['GS_TPDDTEC'])
plot_regression_line(df_gs['Year'], df_gs['ER_mc_functionality_factor'], color=colors['GS_TPDDTEC'], label=None)

# Scatter + regression for fnrb factor (marker='x')
plt.scatter(df_iii_r['Year'], df_iii_r['ER_mc_fnrb_factor'], label='AMS_III_R (FNRB)', marker='x', alpha=0.6, color=colors['AMS_III_R'])
plot_regression_line(df_iii_r['Year'], df_iii_r['ER_mc_fnrb_factor'], color=colors['AMS_III_R'], label=None)

plt.scatter(df_i_e['Year'], df_i_e['ER_mc_fnrb_factor'], label='AMS_I_E (FNRB)', marker='x', alpha=0.6, color=colors['AMS_I_E'])
plot_regression_line(df_i_e['Year'], df_i_e['ER_mc_fnrb_factor'], color=colors['AMS_I_E'], label=None)

plt.scatter(df_gs['Year'], df_gs['ER_mc_fnrb_factor'], label='GS_TPDDTEC (FNRB)', marker='x', alpha=0.6, color=colors['GS_TPDDTEC'])
plot_regression_line(df_gs['Year'], df_gs['ER_mc_fnrb_factor'], color=colors['GS_TPDDTEC'], label=None)


plt.xlabel('Year')
plt.ylabel('Factor Value')
plt.title('Over-crediting Factors over Years by Methodology')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()






### Total ER (Tons of CO2 reduced) per year divided by number of registered households

In [ ]:
df = pd.DataFrame({
    'Year': year, #int
    'Number of non-empty cells': file_non_empty_cells, #int
    'Numeric id': numeric_id, #int
    'Platform': platform, #string identifier
    'ER_mc_functionality_factor': ER_mc_functionality_factor,  #floating point
    'ER_mc_fnrb_factor': ER_mc_fnrb_factor, #floating point
    'AMS_III_R': AMS_III_R, #binary
    'AMS_I_E': AMS_I_E, #binary
    'GS_TPDDTEC': GS_TPDDTEC, #binary
    'ER_per_total_HH': ER_per_total_HH #floating point
})

df = df.dropna(subset=[
    'Year',
    'Number of non-empty cells',
    'ER_mc_functionality_factor',
    'ER_mc_fnrb_factor'
])

# Filter by methodology
df_iii_r = df[df['AMS_III_R'] == 1]
df_i_e = df[df['AMS_I_E'] == 1]
df_gs = df[df['GS_TPDDTEC'] == 1]


def plot_regression_line(x, y, color, label):
    x_reshape = np.array(x).reshape(-1, 1)
    y_reshape = np.array(y).reshape(-1, 1)
    model = LinearRegression()
    model.fit(x_reshape, y_reshape)
    y_pred = model.predict(x_reshape)
    # Sort for cleaner line
    sorted_indices = np.argsort(x)
    plt.plot(np.array(x)[sorted_indices], y_pred[sorted_indices], color=color, linewidth=1, alpha=0.3, label=label)

# Define colors for consistent use
colors = {
    'AMS_III_R': 'tab:blue',
    'AMS_I_E': 'tab:orange',
    'GS_TPDDTEC': 'tab:green'
}


# Create scatter plot
plt.figure(figsize=(12, 6))

# Scatter + regression for functionality factor
plt.scatter(df_iii_r['Year'], df_iii_r['ER_per_total_HH'], label='AMS_III_R', alpha=0.6, color=colors['AMS_III_R'])
plot_regression_line(df_iii_r['Year'], df_iii_r['ER_per_total_HH'], color=colors['AMS_III_R'], label=None)

plt.scatter(df_i_e['Year'], df_i_e['ER_per_total_HH'], label='AMS_I_E', alpha=0.6, color=colors['AMS_I_E'])
plot_regression_line(df_i_e['Year'], df_i_e['ER_per_total_HH'], color=colors['AMS_I_E'], label=None)

plt.scatter(df_gs['Year'], df_gs['ER_per_total_HH'], label='GS_TPDDTEC', alpha=0.6, color=colors['GS_TPDDTEC'])
plot_regression_line(df_gs['Year'], df_gs['ER_per_total_HH'], color=colors['GS_TPDDTEC'], label=None)


plt.xlabel('Year')
plt.ylabel('ER per household')
plt.title('Total ER per year divided by number of registered households')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Data Set Analysis

In [ ]:
df = pd.DataFrame({
    'Year': year, #int
    'Number of non-empty cells': file_non_empty_cells, #int
    'Numeric id': numeric_id, #int
    'Platform': platform, #string identifier
    'ER_mc_functionality_factor': ER_mc_functionality_factor,  #floating point
    'ER_mc_fnrb_factor': ER_mc_fnrb_factor, #floating point
    'AMS_III_R': AMS_III_R, #binary
    'AMS_I_E': AMS_I_E, #binary
    'GS_TPDDTEC': GS_TPDDTEC, #binary
    'ER_per_total_HH': ER_per_total_HH, #floating point
    'fNRB': fNRB #floating point
})


df = df.dropna(subset=[
    'Year',
    'Number of non-empty cells',
    'ER_mc_functionality_factor',
    'ER_mc_fnrb_factor',
    'fNRB'
])

# Filter by methodology
df_iii_r = df[df['AMS_III_R'] == 1]
df_i_e = df[df['AMS_I_E'] == 1]
df_gs = df[df['GS_TPDDTEC'] == 1]


def plot_regression_line(x, y, color, label):
    x_reshape = np.array(x).reshape(-1, 1)
    y_reshape = np.array(y).reshape(-1, 1)
    model = LinearRegression()
    model.fit(x_reshape, y_reshape)
    y_pred = model.predict(x_reshape)
    # Sort for cleaner line
    sorted_indices = np.argsort(x)
    plt.plot(np.array(x)[sorted_indices], y_pred[sorted_indices], color=color, linewidth=1, alpha=0.3, label=label)

# Define colors for consistent use
colors = {
    'AMS_III_R': 'tab:blue',
    'AMS_I_E': 'tab:orange',
    'GS_TPDDTEC': 'tab:green'
}


# Scatter plot of fNRB over Year
plt.figure(figsize=(10, 6))



# Scatter + regression for functionality factor
plt.scatter(df_iii_r['Year'], df_iii_r['fNRB'], label='AMS_III_R', alpha=0.6, color=colors['AMS_III_R'])
plot_regression_line(df_iii_r['Year'], df_iii_r['fNRB'], color=colors['AMS_III_R'], label=None)

plt.scatter(df_i_e['Year'], df_i_e['fNRB'], label='AMS_I_E', alpha=0.6, color=colors['AMS_I_E'])
plot_regression_line(df_i_e['Year'], df_i_e['fNRB'], color=colors['AMS_I_E'], label=None)

plt.scatter(df_gs['Year'], df_gs['fNRB'], label='GS_TPDDTEC', alpha=0.6, color=colors['GS_TPDDTEC'])
plot_regression_line(df_gs['Year'], df_gs['fNRB'], color=colors['GS_TPDDTEC'], label=None)



plt.title('fNRB Over Years')
plt.xlabel('Year')
plt.ylabel('fNRB')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
base

In [ ]:
from scipy.integrate import simps

# fnrb triang function

min_val = 19.1
max_val = 39.6
mode_val = 29.6


base = max_val - min_val
height = 2 / base

# X and Y coordinates of the triangle
x_vals = [min_val, mode_val, max_val]
y_vals = [0, height, 0]

plt.figure(figsize=(6, 6))
plt.plot(x_vals, y_vals, label=f'Triangular Distribution\nmin={min_val}, mode={mode_val}, max={max_val}')
plt.fill(x_vals, y_vals, alpha=0.3)
plt.title('FNRB Triangle Distribution')
plt.xlabel('Value [%]')
plt.ylabel('Density')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
from scipy.integrate import simps

# functionality triang function

min_val = 10
max_val = 90
mode_val = 60


base = max_val - min_val
height = 2 / base

# X and Y coordinates of the triangle
x_vals = [min_val, mode_val, max_val]
y_vals = [0, height, 0]

plt.figure(figsize=(6, 6))
plt.plot(x_vals, y_vals, label=f'Triangular Distribution\nmin={min_val}, mode={mode_val}, max={max_val}')
plt.fill(x_vals, y_vals, alpha=0.3)
plt.title('Functionality Triangle Distribution')
plt.xlabel('Value [%]')
plt.ylabel('Density')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
def plot_triangle(min_val, max_val, mode_val, label):
    base = max_val - min_val
    height = 2 / base
    x_vals = [min_val, mode_val, max_val]
    y_vals = [0, height, 0]
    return x_vals, y_vals, label


# First triangle
#x1, y1, label1 = plot_triangle(10, 92, 60, 'Functionality (10–90, mode=60) (left)')

# Second triangle
x2, y2, label2 = plot_triangle(19.1, 39.6, 29.6, 'FNRB (19.1–39.6, mode=29.6) (left)')

fig, ax1 = plt.subplots(figsize=(10, 7))


#ax1.plot(x1, y1, label=label1)
#ax1.fill_between(x1, y1, alpha=0.3)

ax1.plot(x2, y2, label=label2, color='orange')
ax1.fill_between(x2, y2, alpha=0.3, color='orange')

ax1.set_ylabel("Density (Continuous)")
ax1.set_xlabel("Value [%]")

# Plot
#plt.figure(figsize=(10, 6))
#plt.plot(x1, y1, label=label1)
#plt.fill(x1, y1, alpha=0.3)

#plt.plot(x2, y2, label=label2)
#plt.fill(x2, y2, alpha=0.3)

functionality_discrete_dist = 100 * np.array([0.72,0.60,0.40,0.60,0.40,0.80,0.40,0.80,0.82,0.71,0.63,0.66,0.60,0.40,0.60,0.86,0.95,0.60,0.90,0.0,0.10,0.37,0.69,0.55,0.614,0.69,0.85,0.91,0.92,0.40])

# Create second y-axis for histogram
ax2 = ax1.twinx()

# Plot histogram on ax2
ax2.hist(functionality_discrete_dist, bins=10, edgecolor='black', color='blue', alpha=0.2)
ax2.set_ylabel("Density")

#plt.hist(functionality_discrete_dist,edgecolor='black',color='grey')
#plt.title("Discrete Distribution for Funcionality")
#plt.xlabel("Value [%]")
#plt.ylabel("Frequency")
#plt.show()

ax1.set_ylim(bottom=0)
ax2.set_ylim(bottom=0)
ax1.set_ylim(top=0.12)
ax2.set_ylim(top=9)

plt.title('Triangular and Discrete Distributions for MC simulation')
plt.xlabel('Value [%]')
plt.ylabel('Density (Discrete)')
#plt.grid(True)

hist_patch = plt.Rectangle((0,0),1,1, fc="blue", alpha=0.2, edgecolor='black', label="Histogram")


#plt.legend()
handles, labels = ax1.get_legend_handles_labels()
handles += [hist_patch]
labels += ["Functionality Discrete Distribution (right)"]

# Show combined legend
legend = ax1.legend(handles, labels, loc="upper left", frameon=True)
legend.get_frame().set_facecolor('white')
legend.get_frame().set_edgecolor('black')  

legend = ax1.legend(handles, labels,
                    loc='upper left',
                    #bbox_to_anchor=(0.5, 1.02),
                    frameon=True)

legend.get_frame().set_facecolor('white')
legend.get_frame().set_edgecolor('black')
legend.set_zorder(10)
